In [6]:
from __future__ import annotations

import json
import os
import re
import uuid
import io
from PIL import Image, ImageStat

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
import pandas as pd
import pymupdf as fitz
from openai import OpenAI



import urllib.parse


load_dotenv()

# ============================================================
# Configuration
# ============================================================

INPUT_PDF_DIR = Path("./pdfs")
OUTPUT_DIR = Path("./output")
JSON_DIR = OUTPUT_DIR / "json"
CSV_DIR = OUTPUT_DIR / "csv"
CACHE_DIR = OUTPUT_DIR / "llm_cache"
PAGE_TEXT_DIR = OUTPUT_DIR / "page_text"
EXTRACTED_IMAGES_DIR = OUTPUT_DIR / "extracted_images"
SELECTED_IMAGES_DIR = OUTPUT_DIR / "selected_artifact_images"
SAVE_ALL_EXTRACTED_IMAGES = False
ONLY_FIRST_PDF = False

USE_OPENAI = True
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
MAX_PAGES_PER_RUN: Optional[int] = None  # set None for full run
RETRY_FAILED_PAGES = False

MIN_TEXT_CHARS_FOR_CANDIDATE = 120

PROMPT_TEMPLATE = """
Tu es un archéologue spécialiste du Moyen-Orient ancien, avec une expertise particulière sur le Yémen.

Voici le texte extrait d'une page de PDF :
{full_text}

Liste des images extraites avec leurs centroïdes :
{images_info}

Pour chaque artefact mentionné dans cette page, crée un objet JSON avec les champs suivants :
- numero_inventaire
- type
- description_texte_long
- materiel
- periode
- dimensions
- nom_musee_auction
- lien
- etat
- note
- images_associees

Règles :
- N'inclus que les artefacts réels.
- Chaque artefact doit correspondre à un objet individuel distinct.
- N'extrais pas les phrases générales décrivant une page, une liste ou un ensemble d'objets.
- N'inclus pas une entrée si elle décrit seulement un groupe d'objets sans fiche individuelle identifiable.
- Ne crée pas d'artefact à partir d'une phrase comme "cette liste contient 26 objets" ou de tout autre texte général.
- Ne crée pas d'artefact à partir de "General Organization for Antiquities and Museums" ou de "info@goam.gov.ye".
- Ignore les logos, pages de garde, en-têtes, pieds de page, numéros de page, filigranes, textes institutionnels et images décoratives.
- Reprends les informations dans la langue du document source.
- Ne traduis jamais.
- Si une information est absente, mets null.
- Si une information est incertaine, garde une formulation minimale et factuelle.
- Si une URL (http ou https) est présente, place-la dans le champ `lien`, pas dans `note`.
- Si plusieurs images semblent correspondre, mets une liste.
- Associe les images aux artefacts à partir de la proximité, de l'ordre visuel et des indices textuels.
- Si la page contient seulement un titre, une introduction, une liste générale ou une mention du nombre total d'objets, renvoie une liste vide.
- Renvoie uniquement un JSON valide conforme au schéma demandé.
- Si le texte est trop dégradé pour identifier un objet individuel avec un minimum de certitude, renvoie une liste vide.

""".strip()


# ============================================================
# Data structures
# ============================================================

@dataclass
class ExtractedImage:
    pdf_name: str
    page_num: int
    image_name: str
    centroid_x: float
    centroid_y: float
    image_path: str


@dataclass
class ArtifactRecord:
    artifact_id: str
    pdf_name: str
    page_num: int
    numero_inventaire: Optional[str]
    type: Optional[str]
    description_texte_long: Optional[str]
    materiel: Optional[str]
    periode: Optional[str]
    dimensions: Optional[str]
    nom_musee_auction: Optional[str]
    lien: Optional[str]
    etat: Optional[str]
    note: Optional[str]
    images_associees: List[str]

def interleave_small_large(pdf_files: List[Path]) -> List[Path]:
    files_sorted = sorted(pdf_files, key=lambda p: p.stat().st_size)
    result: List[Path] = []

    left = 0
    right = len(files_sorted) - 1

    while left <= right:
        if left == right:
            result.append(files_sorted[left])
        else:
            result.append(files_sorted[left])   # plus petit restant
            result.append(files_sorted[right])  # plus grand restant
        left += 1
        right -= 1

    return result
    
# ============================================================
# Helpers
# ============================================================

def ensure_dirs() -> None:
    for d in [
        OUTPUT_DIR,
        JSON_DIR,
        CSV_DIR,
        CACHE_DIR,
        PAGE_TEXT_DIR,
        EXTRACTED_IMAGES_DIR,
        SELECTED_IMAGES_DIR,
    ]:
        d.mkdir(parents=True, exist_ok=True)


def normalize_null(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, str):
        s = v.strip()
        if not s or s.lower() in {"null", "none", "n/a", "na"}:
            return None
        return s
    return v


def safe_list(v: Any) -> List[str]:
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v, str) and v.strip():
        return [v.strip()]
    return []


def extract_page_text(page: fitz.Page) -> str:
    blocks = page.get_text("blocks")
    blocks_sorted = sorted(blocks, key=lambda b: (round(b[1], 1), round(b[0], 1)))
    texts: List[str] = []
    for b in blocks_sorted:
        text = (b[4] or "").strip()
        if text:
            texts.append(text)
    return "\n\n".join(texts)


def is_probable_artifact_image(
    image_bytes: bytes,
    rect: fitz.Rect,
    page_rect: fitz.Rect,
    min_w: int = 80,
    min_h: int = 80,
) -> bool:
    # taille affichée sur la page
    disp_w = rect.width
    disp_h = rect.height
    if disp_w < min_w or disp_h < min_h:
        return False

    # éviter header/footer
    page_h = page_rect.height
    if rect.y1 < page_h * 0.12:   # trop haut
        return False
    if rect.y0 > page_h * 0.90:   # trop bas
        return False

    try:
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception:
        return False

    w, h = img.size
    if w < 50 or h < 50:
        return False

    # variance faible = image quasi vide / aplat / masque
    stat = ImageStat.Stat(img)
    variance = sum(stat.var) / 3
    if variance < 15:
        return False

    # trop peu de couleurs distinctes après réduction
    small = img.resize((64, 64))
    colors = small.getcolors(maxcolors=4096)
    if colors is not None and len(colors) < 10:
        return False

    # images ultra allongées souvent décoratives / filets
    ratio = max(w / h, h / w)
    if ratio > 8:
        return False

    return True
    


def extract_images_from_page(doc: fitz.Document, page: fitz.Page, pdf_name: str, page_num: int) -> List[ExtractedImage]:
    images: List[ExtractedImage] = []
    clean_name = clean_pdf_name(pdf_name)
    page_rect = page.rect

    for idx, img in enumerate(page.get_images(full=True), start=1):
        xref = img[0]

        try:
            rects = page.get_image_rects(xref, transform=False)
        except Exception:
            rects = []
        if not rects:
            continue

        rect = rects[0]

        try:
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
        except Exception:
            continue

        if not is_probable_artifact_image(image_bytes, rect, page_rect):
            continue

        image_name = f"{Path(clean_name).stem}_p{page_num:03d}_img{idx:03d}.png"
        image_path = EXTRACTED_IMAGES_DIR / image_name

        if SAVE_ALL_EXTRACTED_IMAGES:
            try:
                image_path.write_bytes(image_bytes)
            except Exception:
                continue

        images.append(
            ExtractedImage(
                pdf_name=clean_name,
                page_num=page_num,
                image_name=image_name,
                centroid_x=round((rect.x0 + rect.x1) / 2, 2),
                centroid_y=round((rect.y0 + rect.y1) / 2, 2),
                image_path=str(image_path) if SAVE_ALL_EXTRACTED_IMAGES else None,
            )
        )

    return images


def page_is_candidate(full_text: str, images: List[ExtractedImage]) -> bool:
    text = full_text.strip()
    if len(text) >= MIN_TEXT_CHARS_FOR_CANDIDATE:
        return True
    if images:
        return True
    return False


def cache_key(pdf_name: str, page_num: int) -> str:
    return f"{Path(pdf_name).stem}_p{page_num:03d}"


def cache_json_path(pdf_name: str, page_num: int) -> Path:
    return CACHE_DIR / f"{cache_key(pdf_name, page_num)}.json"


def failed_marker_path(pdf_name: str, page_num: int) -> Path:
    return CACHE_DIR / f"{cache_key(pdf_name, page_num)}.failed.txt"


def load_cached_artifacts(pdf_name: str, page_num: int) -> Optional[List[Dict[str, Any]]]:
    p = cache_json_path(pdf_name, page_num)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None


def save_cached_artifacts(pdf_name: str, page_num: int, artifacts: List[Dict[str, Any]]) -> None:
    cache_json_path(pdf_name, page_num).write_text(
        json.dumps(artifacts, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    failed = failed_marker_path(pdf_name, page_num)
    if failed.exists():
        failed.unlink()


def should_skip_failed_page(pdf_name: str, page_num: int) -> bool:
    return failed_marker_path(pdf_name, page_num).exists() and not RETRY_FAILED_PAGES


def mark_page_failed(pdf_name: str, page_num: int, message: str) -> None:
    failed_marker_path(pdf_name, page_num).write_text(message, encoding="utf-8")


def build_prompt(full_text: str, images: List[ExtractedImage]) -> str:
    images_info = [
        {
            "image_name": i.image_name,
            "page_num": i.page_num,
            "centroid": {"x": i.centroid_x, "y": i.centroid_y},
        }
        for i in images
    ]
    return PROMPT_TEMPLATE.format(
        full_text=full_text,
        images_info=json.dumps(images_info, ensure_ascii=False, indent=2),
    )


def clean_pdf_name(pdf_name: str) -> str:
    name = Path(pdf_name).name.strip()

    # garder la partie après filename_=UTF-8'' si présente
    marker = "filename_=UTF-8''"
    if marker in name:
        name = name.split(marker, 1)[1]
        name = urllib.parse.unquote(name)
    else:
        # sinon, garder juste avant un éventuel ;
        if ";" in name:
            name = name.split(";", 1)[0].strip()

    return name

def clean_text_field(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, str):
        v = v.replace("\r", " ").replace("\n", " ").replace("\t", " ")
        v = re.sub(r"\s+", " ", v).strip()
        return v or None
    return v

# ============================================================
# LLM extraction
# ============================================================

def call_llm_extract(client: OpenAI, full_text: str, images: List[ExtractedImage], pdf_name: str, page_num: int) -> List[Dict[str, Any]]:
    cached = load_cached_artifacts(pdf_name, page_num)
    if cached is not None:
        print(f"[CACHE] {pdf_name} page {page_num}")
        return cached

    if should_skip_failed_page(pdf_name, page_num):
        print(f"[SKIP FAILED] {pdf_name} page {page_num}")
        return []

    if not USE_OPENAI:
        print(f"[DRY RUN] {pdf_name} page {page_num}")
        return []

    prompt = build_prompt(full_text, images)

    schema = {
        "type": "object",
        "properties": {
            "artifacts": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "numero_inventaire": {"type": ["string", "null"]},
                        "type": {"type": ["string", "null"]},
                        "description_texte_long": {"type": ["string", "null"]},
                        "materiel": {"type": ["string", "null"]},
                        "periode": {"type": ["string", "null"]},
                        "dimensions": {"type": ["string", "null"]},
                        "nom_musee_auction": {"type": ["string", "null"]},
                        "lien": {"type": ["string", "null"]},
                        "etat": {"type": ["string", "null"]},
                        "note": {"type": ["string", "null"]},
                        "images_associees": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "numero_inventaire",
                        "type",
                        "description_texte_long",
                        "materiel",
                        "periode",
                        "dimensions",
                        "nom_musee_auction",
                        "lien",
                        "etat",
                        "note",
                        "images_associees"
                    ],
                    "additionalProperties": False
                }
            }
        },
        "required": ["artifacts"],
        "additionalProperties": False
    }

    try:
        response = client.responses.create(
            model=OPENAI_MODEL,
            input=[
                {
                    "role": "system",
                    "content": [
                        {
                            "type": "input_text",
                            "text": (
                                "You are an archaeologist specializing in ancient Yemen and the ancient Middle East. "
                                "Return only valid JSON matching the schema. "
                                "Do not invent artifacts. Ignore logos, cover pages, decorative images, headers, footers, and unrelated text."
                            ),
                        }
                    ],
                },
                {
                    "role": "user",
                    "content": [{"type": "input_text", "text": prompt}],
                },
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": "artifact_payload",
                    "schema": schema,
                    "strict": True,
                }
            },
        )

        output_text = response.output_text
        parsed = json.loads(output_text)
        artifacts = parsed["artifacts"]
        save_cached_artifacts(pdf_name, page_num, artifacts)
        return artifacts
    except Exception as e:
        mark_page_failed(pdf_name, page_num, repr(e))
        print(f"[ERROR] {pdf_name} page {page_num}: {e}")
        return []


# ============================================================
# Post-processing
# ============================================================

def normalize_artifact(raw: Dict[str, Any], pdf_name: str, page_num: int) -> ArtifactRecord:
    clean_name = clean_pdf_name(pdf_name)

    return ArtifactRecord(
        artifact_id=f"{Path(clean_name).stem}_p{page_num:03d}_{uuid.uuid4().hex[:8]}",
        pdf_name=clean_name,
        page_num=page_num,
        numero_inventaire=clean_text_field(normalize_null(raw.get("numero_inventaire"))),
        type=clean_text_field(normalize_null(raw.get("type"))),
        description_texte_long=clean_text_field(normalize_null(raw.get("description_texte_long"))),
        materiel=clean_text_field(normalize_null(raw.get("materiel"))),
        periode=clean_text_field(normalize_null(raw.get("periode"))),
        dimensions=clean_text_field(normalize_null(raw.get("dimensions"))),
        nom_musee_auction=clean_text_field(normalize_null(raw.get("nom_musee_auction"))),
        lien=clean_text_field(normalize_null(raw.get("lien"))),
        etat=clean_text_field(normalize_null(raw.get("etat"))),
        note=clean_text_field(normalize_null(raw.get("note"))),
        images_associees=[clean_text_field(x) for x in safe_list(raw.get("images_associees")) if clean_text_field(x)],
    )


def artifacts_to_dataframe(artifacts: List[ArtifactRecord]) -> pd.DataFrame:
    return pd.DataFrame([asdict(a) for a in artifacts])


def images_to_dataframe(artifacts: List[ArtifactRecord], extracted_images: List[ExtractedImage]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    image_lookup = {img.image_name: img for img in extracted_images}

    for artifact in artifacts:
        for image_name in artifact.images_associees:
            img = image_lookup.get(image_name)
            rows.append(
                {
                    "artifact_id": artifact.artifact_id,
                    "pdf_name": artifact.pdf_name,
                    "page_num": artifact.page_num,
                    "image_name": image_name,
                    "image_path": img.image_path if img else None,
                    "centroid_x": img.centroid_x if img else None,
                    "centroid_y": img.centroid_y if img else None,
                }
            )

    return pd.DataFrame(rows)

def save_selected_images(doc: fitz.Document, artifacts: List[ArtifactRecord]) -> None:
    SELECTED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

    clean_name = clean_pdf_name(doc.name)
    pdf_artifacts = [a for a in artifacts if a.pdf_name == clean_name]

    if not pdf_artifacts:
        return

    needed_images = set()
    for artifact in pdf_artifacts:
        needed_images.update(artifact.images_associees)

    if not needed_images:
        return

    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        page_num = page_index + 1

        for idx, img in enumerate(page.get_images(full=True), start=1):
            xref = img[0]
            image_name = f"{Path(clean_name).stem}_p{page_num:03d}_img{idx:03d}.png"

            if image_name not in needed_images:
                continue

            output_path = SELECTED_IMAGES_DIR / image_name
            if output_path.exists():
                continue

            try:
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                output_path.write_bytes(image_bytes)
            except Exception as e:
                print(f"[WARN] Could not save selected image {image_name}: {e}")
                
# ============================================================
# Pipeline
# ============================================================

def process_pdf(client: OpenAI, pdf_path: Path) -> Tuple[List[ArtifactRecord], List[ExtractedImage]]:
    doc = fitz.open(pdf_path)
    pdf_name = pdf_path.name
    artifacts: List[ArtifactRecord] = []
    all_images: List[ExtractedImage] = []
    processed_candidates = 0

    for page_index in range(len(doc)):
        if MAX_PAGES_PER_RUN is not None and processed_candidates >= MAX_PAGES_PER_RUN:
            break

        page = doc.load_page(page_index)
        page_num = page_index + 1
        full_text = extract_page_text(page)
        (PAGE_TEXT_DIR / f"{cache_key(pdf_name, page_num)}.txt").write_text(full_text, encoding="utf-8")

        images = extract_images_from_page(doc, page, pdf_name, page_num)
        all_images.extend(images)

        if not page_is_candidate(full_text, images):
            print(f"[SKIP] {pdf_name} page {page_num}")
            continue

        processed_candidates += 1
        raw_artifacts = call_llm_extract(client, full_text, images, pdf_name, page_num)

        for raw in raw_artifacts:
            artifacts.append(normalize_artifact(raw, pdf_name, page_num))

    return artifacts, all_images


def run_pipeline() -> None:
    ensure_dirs()

    api_key = os.getenv("OPENAI_API_KEY")
    if USE_OPENAI and not api_key:
        raise ValueError("OPENAI_API_KEY is missing")

    client = OpenAI(api_key=api_key) if USE_OPENAI else OpenAI(api_key="dummy")

    pdf_files = list(INPUT_PDF_DIR.glob("*.pdf"))
    pdf_files = interleave_small_large(pdf_files)
    for p in pdf_files:
        print(f"{p.name} | {round(p.stat().st_size / (1024 * 1024), 2)} MB")
    
    if ONLY_FIRST_PDF:
        pdf_files = pdf_files[:1]
    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in {INPUT_PDF_DIR.resolve()}")

    all_artifacts: List[ArtifactRecord] = []
    all_extracted_images: List[ExtractedImage] = []

    for pdf_path in pdf_files:
        print(f"[INFO] Processing {pdf_path.name}")
        artifacts, extracted_images = process_pdf(client, pdf_path)
        all_artifacts.extend(artifacts)
        all_extracted_images.extend(extracted_images)

    artifacts_df = artifacts_to_dataframe(all_artifacts)
    images_df = images_to_dataframe(all_artifacts, all_extracted_images)

    artifacts_json_path = JSON_DIR / "all_artifacts.json"
    artifacts_csv_path = CSV_DIR / "artifacts.csv"
    images_csv_path = CSV_DIR / "artifact_images.csv"

    artifacts_json_path.write_text(
        json.dumps([asdict(a) for a in all_artifacts], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if not artifacts_df.empty:
        artifacts_df["images_associees"] = artifacts_df["images_associees"].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
        )

    artifacts_df.to_csv(artifacts_csv_path, index=False, encoding="utf-8-sig")
    images_df.to_csv(images_csv_path, index=False, encoding="utf-8-sig")

    for pdf_path in pdf_files:
        doc = fitz.open(pdf_path)
        save_selected_images(doc, all_artifacts)

    print("[DONE]")
    print(f"Artifacts JSON: {artifacts_json_path}")
    print(f"Artifacts CSV:  {artifacts_csv_path}")
    print(f"Images CSV:     {images_csv_path}")
    print(f"Extracted imgs: {EXTRACTED_IMAGES_DIR}")
    print(f"Selected imgs:  {SELECTED_IMAGES_DIR}")


if __name__ == "__main__":
    run_pipeline()


0-______ _________18.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_18.pdf | 0.75 MB
0-______ _________25.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_25.pdf | 4.05 MB
0-______ _________7.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_7.pdf | 0.8 MB
0-______ _________10.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_10.pdf | 3.78 MB
0-______ _________16.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_16.pdf | 0.85 MB
0-______ _________20.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_20.pdf | 3.21 MB
0-______ _________5.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_5.pdf | 1.1 MB
0-______ _________24.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_24.pdf | 2.68 MB
31-______ _________31.pdf_; filename_=UTF-8''31-آثارنا المنهوبة_31.pdf | 1.18 MB
0-______ _________19.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_19.pdf | 2.64 MB
0-______ _________3.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_3.pdf | 1.19 MB
0-______ _________23.pdf_; filename_=UTF-8''0-آثارنا المنهوبة_23.pdf | 2.54 MB
0-______ _________21.pdf_; filename_=UTF-8''0-آثارنا المنه